### Init


In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim,col

### Reading Bronze data - cust az12

In [0]:
df = spark.table("workspace.bronze.erp_cust_az12")

In [0]:
df.display()

  ### Data  Transformation

  ##Trimming

In [0]:
for item in  df.schema.fields:
  if isinstance(item.dataType,StringType):
    df = df.withColumn(item.name,trim(col(item.name)))


  ##CustomerID CleanUp

In [0]:
df = df.withColumn(
    "cid",
    F.when(col("cid").startswith("NAS"),
           F.substring(col("cid"), 4, F.length(col("cid"))))
     .otherwise(col("cid"))
)

  ##BirthDate Validation

In [0]:
df = (
    df.withColumn(
        "BDATE",
        F.when(col("BDATE") > F.current_date(), None)
        .otherwise(col("BDATE"))
    )
)

  ##Gender Normalization

In [0]:
df = df.withColumn(
    "gen",
    F.when(F.upper(col("gen")).isin("F", "FEMALE"), "Female")
     .when(F.upper(col("gen")).isin("M", "MALE"), "Male")
     .otherwise("n/a")
)

In [0]:
  ##Data Dictionary
RENAME_MAP = {
    "cid": "customer_number",
    "bdate": "birth_date",
    "gen": "gender"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)
df.display()

  ##Writing Silver table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.erp_customers")